# Feature Engineering — MEPS 2022

**Goal**: Transform the 48 raw MEPS columns into a clean, fully-encoded, model-ready feature matrix.
Age-gated logic resolves the survey-design missingness identified in `02_eda.ipynb`.

## Age-Gating Rules
| Feature | MEPS restriction | Rule |
|---|---|---|
| `bmi` (unified) | adult bmi = 18+; child_bmi = 6-17 | Merge into one column; impute by age-group median |
| `is_student` | 17-23 only | Outside range -> 0 (not a student) |
| `k6_distress_score`, `phq2_depression_score` | SAQ, adults 18+ | Under 18 -> 0; adult non-responders -> adult median |
| Adult chronic conditions (9 flags) | 18+ only | Under 18 -> 0 |
| `dx_adhd_add` | 5-17 only | Outside range -> 0 |
| `employment_status` | 16+ | Under 16 -> 0 (N/A) |
| `education_years` | 5+ | Under 5 -> 0 (too young for school) |

## Features Dropped
| Feature | Reason |
|---|---|
| `arthritis_type` | 76% missing; `dx_arthritis` yes/no is sufficient |
| `child_bmi` | Merged into unified `bmi` |
| `wage_income`, `business_income` | Components of `total_person_income`; avoid multicollinearity |
| `insurance_type`, `insured_full_year` | Derived from the 6 binary insurance flags; redundant |
| `person_id` | ID column, not a feature |

## Output Files
- `X_train.csv` / `X_test.csv` — unscaled (Random Forest, XGBoost)
- `X_train_scaled.csv` / `X_test_scaled.csv` — StandardScaler (Linear Regression, SVR)
- `y_train.csv` / `y_test.csv` — log1p(total_healthcare_expenditure)
- `scaler.pkl` — fitted StandardScaler for the web app prediction pipeline

In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Load & Clean

Identical pipeline to `02_eda.ipynb`: select the 48 target columns, rename, replace MEPS reserved codes with NaN.

In [2]:
meps_raw = pd.read_stata('h243.dta')

wanted_cols = [
    'DUPERSID',
    'AGE22X', 'SEX', 'RACEV2X', 'HISPANX', 'MARRY22X',
    'EDUCYR', 'FTSTU22X', 'REGION22',
    'POVCAT22', 'FAMINC22', 'TTLP22X', 'WAGEP22X', 'BUSNP22X', 'EMPST53',
    'RTHLTH53', 'MNHLTH53', 'IADLHP31', 'ADLHLP31',
    'ANYLMI22', 'K6SUM42', 'PHQ242',
    'HIBPDX', 'CHDDX', 'ANGIDX', 'MIDX', 'STRKDX', 'EMPHDX',
    'CHOLDX', 'CANCERDX', 'ARTHDX', 'ARTHTYPE', 'ASTHDX',
    'ADHDADDX', 'DIABDX_M18',
    'INSCOV22', 'INSURC22', 'PRVEV22', 'TRIEV22',
    'MCREV22', 'MCDEV22', 'VAEV22', 'UNINS22',
    'ADBMI42', 'CHBMIX42', 'HAVEUS42', 'FAMSZE22',
    'TOTEXP22',
]
model_cols = [c for c in wanted_cols if c in meps_raw.columns]
meps = meps_raw[model_cols].copy()

rename_map = {
    'DUPERSID': 'person_id',
    'AGE22X': 'age', 'SEX': 'sex', 'RACEV2X': 'race',
    'HISPANX': 'hispanic', 'MARRY22X': 'marital_status',
    'EDUCYR': 'education_years', 'FTSTU22X': 'is_student',
    'REGION22': 'region_2022',
    'POVCAT22': 'poverty_category', 'FAMINC22': 'family_income',
    'TTLP22X': 'total_person_income', 'WAGEP22X': 'wage_income',
    'BUSNP22X': 'business_income', 'EMPST53': 'employment_status',
    'RTHLTH53': 'self_rated_health', 'MNHLTH53': 'self_rated_mental_health',
    'IADLHP31': 'needs_help_iadl', 'ADLHLP31': 'needs_help_adl',
    'ANYLMI22': 'any_mental_illness',
    'K6SUM42': 'k6_distress_score', 'PHQ242': 'phq2_depression_score',
    'HIBPDX': 'dx_hypertension', 'CHDDX': 'dx_coronary_heart_disease',
    'ANGIDX': 'dx_angina', 'MIDX': 'dx_myocardial_infarction',
    'STRKDX': 'dx_stroke', 'EMPHDX': 'dx_emphysema',
    'CHOLDX': 'dx_high_cholesterol', 'CANCERDX': 'dx_cancer',
    'ARTHDX': 'dx_arthritis', 'ARTHTYPE': 'arthritis_type',
    'ASTHDX': 'dx_asthma', 'ADHDADDX': 'dx_adhd_add',
    'DIABDX_M18': 'dx_diabetes',
    'INSCOV22': 'insured_full_year', 'INSURC22': 'insurance_type',
    'PRVEV22': 'has_private_insurance', 'TRIEV22': 'has_tricare',
    'MCREV22': 'has_medicare', 'MCDEV22': 'has_medicaid',
    'VAEV22': 'has_va_coverage', 'UNINS22': 'uninsured_status',
    'ADBMI42': 'bmi', 'CHBMIX42': 'child_bmi',
    'HAVEUS42': 'has_usual_care', 'FAMSZE22': 'family_size',
    'TOTEXP22': 'total_healthcare_expenditure',
}
meps = meps.rename(columns=rename_map)

# Replace MEPS reserved codes with NaN
reserved_str = ['-1 INAPPLICABLE', '-7 REFUSED', '-15 CANNOT BE COMPUTED']
reserved_str.append("-8 DON'T KNOW")   # apostrophe requires double-quoted Python string
reserved_num = [-1, -2, -7, -8, -9, -10, -13, -15]

df = meps.copy()
for col in df.columns:
    if isinstance(df[col].dtype, pd.CategoricalDtype):
        df[col] = df[col].astype(object).replace(reserved_str, np.nan)
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].replace(reserved_num, np.nan)
for col in ['bmi', 'child_bmi', 'k6_distress_score', 'phq2_depression_score']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# age_ref: age with NaN filled by median — used for all age-gating masks below
age_ref = df['age'].fillna(df['age'].median())

print(f'Loaded : {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Total NaN cells after cleaning: {df.isnull().sum().sum():,}')

Loaded : 22,431 rows x 48 columns
Total NaN cells after cleaning: 163,005


## 2. Age-Gated Feature Engineering

MEPS survey design means missing values are *by design*, not data-quality errors.
Each feature below is resolved using the respondent's age.

The same decision tree is used in the web app:
the front-end conditionally shows/hides questions based on the user's entered age.

In [3]:
# ── 1. Unified BMI ─────────────────────────────────────────────────────────────
# adult bmi (18+) and child_bmi (6-17) are mutually exclusive in MEPS.
# Merge them into one column so the website asks a single BMI question for all ages.
bmi_unified = df['bmi'].copy()
child_mask = (age_ref >= 6) & (age_ref < 18)
bmi_unified.loc[child_mask] = df.loc[child_mask, 'child_bmi']

print('Unified BMI coverage:')
print(f'  Under 6   ({(age_ref < 6).sum():>5,} rows): {bmi_unified[age_ref < 6].notnull().sum()} present -> will median-impute')
print(f'  Ages 6-17 ({child_mask.sum():>5,} rows): {bmi_unified[child_mask].notnull().sum():,} present ({bmi_unified[child_mask].notnull().mean()*100:.1f}%)')
print(f'  Ages 18+  ({(age_ref >= 18).sum():>5,} rows): {bmi_unified[age_ref >= 18].notnull().sum():,} present ({bmi_unified[age_ref >= 18].notnull().mean()*100:.1f}%)')
print()

# ── 2. is_student (0=not student or N/A, 1=part-time, 2=full-time) ─────────────
# MEPS only collects this for ages 17-23; all others were INAPPLICABLE -> NaN.
# Web app: show this question only when user enters age 17-23.
student_map = {
    '1 FULL-TIME STUDENT': 2,
    '2 PART-TIME STUDENT': 1,
    '3 NOT A STUDENT':     0,
}
is_student = df['is_student'].map(student_map).fillna(0).astype(int)
print(f'is_student: full-time={( is_student==2).sum():,}  part-time={(is_student==1).sum():,}  not/NA={(is_student==0).sum():,}')
print()

# ── 3. Mental health scores (SAQ, adults 18+ only) ─────────────────────────────
# Under 18: 0 (not assessed by design).
# Adults with SAQ missing (~40% of 18+): impute with adult median.
# Web app: show k6 and phq2 questions only when age >= 18.
k6   = df['k6_distress_score'].copy()
phq2 = df['phq2_depression_score'].copy()

k6.loc[age_ref < 18]   = 0.0
phq2.loc[age_ref < 18] = 0.0

k6_adult_med   = k6[age_ref >= 18].median()
phq2_adult_med = phq2[age_ref >= 18].median()
k6   = k6.fillna(k6_adult_med)
phq2 = phq2.fillna(phq2_adult_med)

print(f'k6_distress_score     NaN remaining: {k6.isnull().sum()}  (adult median = {k6_adult_med})')
print(f'phq2_depression_score NaN remaining: {phq2.isnull().sum()}  (adult median = {phq2_adult_med})')
print()

# ── 4. Chronic conditions ─────────────────────────────────────────────────────
# Adult-only (18+): under 18 -> 0; remaining adult NaN -> 0 (assume no diagnosis)
# ADHD/ADD (5-17 only): outside range -> 0
# All-ages (asthma, diabetes): NaN -> 0
# Web app: adult dx questions shown only when age >= 18; dx_adhd_add shown for ages 5-17.
def yesno_enc(series):
    return series.map({'1 YES': 1, '2 NO': 0})

adult_dx_cols = [
    'dx_hypertension', 'dx_coronary_heart_disease', 'dx_angina',
    'dx_myocardial_infarction', 'dx_stroke', 'dx_emphysema',
    'dx_high_cholesterol', 'dx_cancer', 'dx_arthritis',
]
dx = {}
for col in adult_dx_cols:
    enc = yesno_enc(df[col])
    enc.loc[age_ref < 18] = 0
    dx[col] = enc.fillna(0).astype(int)

# ADHD/ADD: ages 5-17 only
adhd = yesno_enc(df['dx_adhd_add'])
adhd.loc[age_ref < 5]  = 0
adhd.loc[age_ref > 17] = 0
dx['dx_adhd_add'] = adhd.fillna(0).astype(int)

dx['dx_asthma']  = yesno_enc(df['dx_asthma']).fillna(0).astype(int)
dx['dx_diabetes'] = yesno_enc(df['dx_diabetes']).fillna(0).astype(int)

print('Diagnosis prevalences (after age-gating):')
for name, enc in dx.items():
    print(f'  {name:<35} {enc.mean()*100:5.1f}%  NaN: {enc.isnull().sum()}')
print()

# ── 5. Employment status (0=under-16 or N/A; 1-4=MEPS categories) ─────────────
emp_map = {
    '1 EMPLOYED AT RD 5/3 INT DATE':           1,
    '2 JOB TO RETURN TO AT RD 5/3 INT DATE':   2,
    '3 JOB DURING RD 5/3 REF PERIOD':          3,
    '4 NOT EMPLOYED DURING RD 5/3':            4,
}
employment = df['employment_status'].map(emp_map).fillna(0).astype(int)

# ── 6. Education years (under 5 -> 0; refused among 5+ -> mode) ───────────────
def parse_edu(val):
    if pd.isna(val): return np.nan
    try: return int(str(val).strip().split()[0])
    except: return np.nan

education = df['education_years'].apply(parse_edu)
education.loc[age_ref < 5] = 0
edu_mode = int(education[age_ref >= 5].mode()[0])
education = education.fillna(edu_mode).astype(int)

print(f'employment_status NaN: {employment.isnull().sum()}')
print(f'education_years   NaN: {education.isnull().sum()}  (mode for ages 5+: {edu_mode})')

Unified BMI coverage:
  Under 6   (1,200 rows): 0 present -> will median-impute
  Ages 6-17 (3,127 rows): 2,266 present (72.5%)
  Ages 18+  (18,104 rows): 10,601 present (58.6%)

is_student: full-time=827  part-time=104  not/NA=21,500

k6_distress_score     NaN remaining: 0  (adult median = 2.0)
phq2_depression_score NaN remaining: 0  (adult median = 0.0)

Diagnosis prevalences (after age-gating):
  dx_hypertension                      30.3%  NaN: 0
  dx_coronary_heart_disease             4.9%  NaN: 0
  dx_angina                             1.7%  NaN: 0
  dx_myocardial_infarction              3.3%  NaN: 0
  dx_stroke                             4.0%  NaN: 0
  dx_emphysema                          1.6%  NaN: 0
  dx_high_cholesterol                  28.1%  NaN: 0
  dx_cancer                            10.5%  NaN: 0
  dx_arthritis                         23.6%  NaN: 0
  dx_adhd_add                           2.0%  NaN: 0
  dx_asthma                            13.8%  NaN: 0
  dx_diabetes   

## 3. Encoding Categorical Features

| Encoding | Features |
|---|---|
| Binary 0/1 | sex, hispanic, all yes/no flags, insurance flags |
| Ordinal | self_rated_health (1-5), self_rated_mental_health (1-5), poverty_category (1-5) |
| One-hot | race (8 cats), marital_status (6 cats), region (4 cats) |
| Signed log1p | family_income, total_person_income (handles negative values) |

In [4]:
# ── Helpers ────────────────────────────────────────────────────────────────────
def yesno(series, fill=0):
    return series.map({'1 YES': 1, '2 NO': 0}).fillna(fill).astype(int)

def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))

def parse_leading_int(val):
    if pd.isna(val): return np.nan
    try: return int(str(val).strip().split()[0])
    except: return np.nan

# ── Demographics: binary ───────────────────────────────────────────────────────
sex      = df['sex'].map({'1 MALE': 1, '2 FEMALE': 0}).astype(int)
hispanic = df['hispanic'].map({'1 HISPANIC': 1, '2 NOT HISPANIC': 0}).astype(int)

# ── Ordinal: self-rated health (1=Excellent ... 5=Poor) ───────────────────────
hmap = {'1 EXCELLENT': 1, '2 VERY GOOD': 2, '3 GOOD': 3, '4 FAIR': 4, '5 POOR': 5}
self_health  = df['self_rated_health'].map(hmap)
self_mhealth = df['self_rated_mental_health'].map(hmap)

# ── Ordinal: poverty (1=Poor/Negative ... 5=High Income) ─────────────────────
pmap = {'1 POOR/NEGATIVE': 1, '2 NEAR POOR': 2, '3 LOW INCOME': 3,
        '4 MIDDLE INCOME': 4, '5 HIGH INCOME': 5}
poverty = df['poverty_category'].map(pmap).astype(int)

# ── Numeric: family_size ──────────────────────────────────────────────────────
family_size = df['family_size'].apply(parse_leading_int)

# ── Income: signed log1p preserves sign and compresses magnitude ──────────────
# Handles negative incomes; $0 stays $0; symmetric around origin.
family_income_tr       = df['family_income'].apply(signed_log1p)
total_person_income_tr = df['total_person_income'].apply(signed_log1p)

# ── Health utilization & limitation: binary ───────────────────────────────────
needs_help_iadl = yesno(df['needs_help_iadl'])
needs_help_adl  = yesno(df['needs_help_adl'])
# ANYLMI22 = any IADL/ADL/functional limitation (variable name is misleading)
any_limitation  = yesno(df['any_mental_illness'])

# ── Insurance: binary (all flags are 0% missing in source data) ───────────────
has_private_ins = yesno(df['has_private_insurance'])
has_tricare     = yesno(df['has_tricare'])
has_medicare    = yesno(df['has_medicare'])
has_medicaid    = yesno(df['has_medicaid'])
has_va          = yesno(df['has_va_coverage'])
is_uninsured    = yesno(df['uninsured_status'])
has_usual_care  = yesno(df['has_usual_care'])

# ── One-hot: Race ──────────────────────────────────────────────────────────────
race_dum = pd.get_dummies(df['race'].fillna('UNKNOWN'), prefix='race').astype(int)
race_ren = {}
for c in race_dum.columns:
    lc = c.lower()
    if 'white' in lc:                              race_ren[c] = 'race_white'
    elif 'black' in lc:                            race_ren[c] = 'race_black'
    elif 'amer ind' in lc or 'alaska' in lc:       race_ren[c] = 'race_native_american'
    elif 'asian ind' in lc:                        race_ren[c] = 'race_asian_indian'
    elif 'chinese' in lc:                          race_ren[c] = 'race_chinese'
    elif 'filipin' in lc:                          race_ren[c] = 'race_filipino'
    elif 'multiple' in lc:                         race_ren[c] = 'race_multiple'
    elif 'oth asian' in lc or 'pacfc' in lc:       race_ren[c] = 'race_other_asian_pi'
    else:                                          race_ren[c] = 'race_unknown'
race_dum = race_dum.rename(columns=race_ren)
if 'race_unknown' in race_dum.columns:
    race_dum = race_dum.drop(columns=['race_unknown'])   # 0% missing -> all-zero column

# ── One-hot: Marital status ────────────────────────────────────────────────────
# '6 UNDER AGE 16 - INAPPLICABLE' is a real MEPS category value (not cleaned to NaN)
marital_str = {
    '1 MARRIED': 'married', '2 WIDOWED': 'widowed', '3 DIVORCED': 'divorced',
    '4 SEPARATED': 'separated', '5 NEVER MARRIED': 'never_married',
    '6 UNDER AGE 16 - INAPPLICABLE': 'under_16',
}
marital_dum = pd.get_dummies(
    df['marital_status'].map(marital_str).fillna('never_married'),
    prefix='marital').astype(int)

# ── One-hot: Region (195 NaN -> south, the mode) ──────────────────────────────
region_str = {'1 NORTHEAST': 'northeast', '2 MIDWEST': 'midwest',
              '3 SOUTH': 'south', '4 WEST': 'west'}
region_dum = pd.get_dummies(
    df['region_2022'].map(region_str).fillna('south'),
    prefix='region').astype(int)

print(f'Race columns    : {list(race_dum.columns)}')
print(f'Marital columns : {list(marital_dum.columns)}')
print(f'Region columns  : {list(region_dum.columns)}')

Race columns    : ['race_white', 'race_other_asian_pi', 'race_multiple', 'race_black', 'race_native_american', 'race_asian_indian', 'race_chinese', 'race_filipino']
Marital columns : ['marital_divorced', 'marital_married', 'marital_never_married', 'marital_separated', 'marital_under_16', 'marital_widowed']
Region columns  : ['region_midwest', 'region_northeast', 'region_south', 'region_west']


## 4. Residual NaN Imputation

After age-gating and encoding, a small number of NaN values remain.
All are imputed so every model receives a complete feature matrix.
The imputation values should be saved as defaults in the web app.

In [5]:
# ── age: 195 NaN -> median ─────────────────────────────────────────────────────
age_med = int(df['age'].median())
age_imputed = df['age'].fillna(age_med).astype(int)

# ── BMI: impute by age group with that group's median ─────────────────────────
# Under 6: MEPS collects no BMI -> use overall dataset median as proxy
# Ages 6-17 and 18+: use their respective medians
bmi = bmi_unified.copy()
bmi_overall_med = float(bmi.dropna().median())
bmi_child_med   = float(bmi[(age_ref >= 6) & (age_ref < 18)].median())
bmi_adult_med   = float(bmi[age_ref >= 18].median())

bmi.loc[(age_ref < 6)  & bmi.isnull()] = bmi_overall_med
bmi.loc[(age_ref >= 6) & (age_ref < 18) & bmi.isnull()] = bmi_child_med
bmi.loc[(age_ref >= 18) & bmi.isnull()] = bmi_adult_med
bmi = bmi.round(1)

# ── Self-rated health: median impute ──────────────────────────────────────────
sh_med  = int(self_health.median())
smh_med = int(self_mhealth.median())
self_health_imp  = self_health.fillna(sh_med).astype(int)
self_mhealth_imp = self_mhealth.fillna(smh_med).astype(int)

# ── Family size: mode impute (204 NaN) ────────────────────────────────────────
fs_mode = int(family_size.mode()[0])
family_size_imp = family_size.fillna(fs_mode).astype(int)

print('Imputation summary:')
print(f'  age                : 195 NaN -> median = {age_med}')
print(f'  bmi (under 6)      : {(age_ref < 6).sum()} rows -> overall median = {bmi_overall_med:.1f}')
print(f'  bmi (ages 6-17)    : child median = {bmi_child_med:.1f}')
print(f'  bmi (ages 18+)     : adult median = {bmi_adult_med:.1f}')
print(f'  bmi NaN remaining  : {bmi.isnull().sum()}')
print(f'  self_health        : {self_health.isnull().sum()} NaN -> median = {sh_med}')
print(f'  self_mhealth       : {self_mhealth.isnull().sum()} NaN -> median = {smh_med}')
print(f'  family_size        : {family_size.isnull().sum()} NaN -> mode = {fs_mode}')

Imputation summary:
  age                : 195 NaN -> median = 44
  bmi (under 6)      : 1200 rows -> overall median = 26.5
  bmi (ages 6-17)    : child median = 20.3
  bmi (ages 18+)     : adult median = 27.4
  bmi NaN remaining  : 0
  self_health        : 358 NaN -> median = 2
  self_mhealth       : 367 NaN -> median = 2
  family_size        : 204 NaN -> mode = 2


## 5. Assemble Final Feature DataFrame

Combine all engineered features into one DataFrame.
Every column is numeric (int or float). No NaN values remain.

In [6]:
final_df = pd.DataFrame({
    # ── Demographics ──────────────────────────────────────────────────────────
    'age':                      age_imputed,
    'sex':                      sex,               # 1=male, 0=female
    'hispanic':                 hispanic,          # 1=hispanic, 0=not hispanic
    'bmi':                      bmi,               # unified: child_bmi for 6-17, adult bmi for 18+

    # ── Socioeconomic ─────────────────────────────────────────────────────────
    'education_years':          education,         # 0-17 (0 for under 5)
    'family_size':              family_size_imp,   # 1-14
    'poverty_category':         poverty,           # 1=poor/negative ... 5=high income
    'family_income':            family_income_tr,  # signed log1p transformed
    'total_person_income':      total_person_income_tr,
    'employment_status':        employment,        # 0=N/A(<16), 1=employed ... 4=not employed
    'is_student':               is_student,        # 0=not/NA, 1=part-time, 2=full-time

    # ── Health Status ─────────────────────────────────────────────────────────
    'self_rated_health':        self_health_imp,   # 1=excellent ... 5=poor
    'self_rated_mental_health': self_mhealth_imp,
    'needs_help_iadl':          needs_help_iadl,   # 1=yes, 0=no
    'needs_help_adl':           needs_help_adl,
    'any_limitation':           any_limitation,    # any IADL/ADL/functional limitation
    'k6_distress_score':        k6.round(2),       # 0-24 (0 for under 18)
    'phq2_depression_score':    phq2.round(2),     # 0-6  (0 for under 18)

    # ── Chronic Conditions ────────────────────────────────────────────────────
    'dx_hypertension':             dx['dx_hypertension'],
    'dx_coronary_heart_disease':   dx['dx_coronary_heart_disease'],
    'dx_angina':                   dx['dx_angina'],
    'dx_myocardial_infarction':    dx['dx_myocardial_infarction'],
    'dx_stroke':                   dx['dx_stroke'],
    'dx_emphysema':                dx['dx_emphysema'],
    'dx_high_cholesterol':         dx['dx_high_cholesterol'],
    'dx_cancer':                   dx['dx_cancer'],
    'dx_arthritis':                dx['dx_arthritis'],
    'dx_asthma':                   dx['dx_asthma'],
    'dx_adhd_add':                 dx['dx_adhd_add'],
    'dx_diabetes':                 dx['dx_diabetes'],

    # ── Insurance ─────────────────────────────────────────────────────────────
    'has_private_insurance':   has_private_ins,
    'has_tricare':             has_tricare,
    'has_medicare':            has_medicare,
    'has_medicaid':            has_medicaid,
    'has_va_coverage':         has_va,
    'is_uninsured':            is_uninsured,
    'has_usual_care':          has_usual_care,
}, index=df.index)

# Concatenate one-hot encoded columns
final_df = pd.concat([final_df, race_dum, marital_dum, region_dum], axis=1)

# Target: log1p(total annual healthcare expenditure)
final_df['target'] = np.log1p(df['total_healthcare_expenditure'])

print(f'Final shape  : {final_df.shape}')
print(f'Feature cols : {final_df.shape[1] - 1}')
print(f'Target NaN   : {final_df["target"].isnull().sum()}')
print(f'Total NaN    : {final_df.isnull().sum().sum()}')

Final shape  : (22431, 56)
Feature cols : 55
Target NaN   : 0
Total NaN    : 0


In [7]:
# ── NaN check ─────────────────────────────────────────────────────────────────
nan_check = final_df.isnull().sum()
nan_check = nan_check[nan_check > 0]
if len(nan_check) == 0:
    print('No remaining NaN values — feature matrix is complete.')
else:
    print('WARNING — columns with remaining NaN:')
    print(nan_check)

# ── Feature overview ──────────────────────────────────────────────────────────
feature_cols = [c for c in final_df.columns if c != 'target']
print(f'\nFeature summary ({len(feature_cols)} columns):')
print(final_df[feature_cols].describe().T[['mean', 'std', 'min', 'max']].round(3).to_string())

No remaining NaN values — feature matrix is complete.

Feature summary (55 columns):
                             mean     std     min     max
age                        43.355  23.643   0.000  85.000
sex                         0.473   0.499   0.000   1.000
hispanic                    0.218   0.413   0.000   1.000
bmi                        27.042   5.544  10.200  50.000
education_years            11.549   4.914   0.000  17.000
family_size                 2.960   1.667   1.000  14.000
poverty_category            3.639   1.435   1.000   5.000
family_income              10.609   2.372 -13.362  13.577
total_person_income         7.691   4.670 -12.717  13.376
employment_status           1.861   1.587   0.000   4.000
is_student                  0.078   0.382   0.000   2.000
self_rated_health           2.310   1.031   1.000   5.000
self_rated_mental_health    2.213   1.008   1.000   5.000
needs_help_iadl             0.036   0.186   0.000   1.000
needs_help_adl              0.021   0.143   0

## 6. Train / Test Split & Save

**80/20 split** with `random_state=42` for reproducibility.

Two versions are saved:
- **Unscaled** — for Random Forest & XGBoost (scale-invariant tree models)
- **Scaled** — StandardScaler fitted on train only, for Linear Regression & SVR

The fitted scaler is saved to `scaler.pkl` for use in the web app's prediction pipeline.

In [8]:
feature_cols = [c for c in final_df.columns if c != 'target']
X = final_df[feature_cols]
y = final_df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f'Train : {X_train.shape[0]:,} rows  ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test  : {X_test.shape[0]:,} rows  ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'Features: {X_train.shape[1]}')
print(f'\nTarget (log1p expenditure):')
print(f'  Train  mean={y_train.mean():.3f}  std={y_train.std():.3f}  min={y_train.min():.1f}  max={y_train.max():.1f}')
print(f'  Test   mean={y_test.mean():.3f}  std={y_test.std():.3f}  min={y_test.min():.1f}  max={y_test.max():.1f}')

Train : 17,944 rows  (80%)
Test  : 4,487 rows  (20%)
Features: 55

Target (log1p expenditure):
  Train  mean=6.563  std=3.215  min=0.0  max=14.1
  Test   mean=6.591  std=3.213  min=0.0  max=12.7


In [9]:
# ── StandardScaler: fit on train only to prevent data leakage ─────────────────
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=feature_cols, index=X_train.index)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=feature_cols, index=X_test.index)

# ── Save all CSVs and scaler ───────────────────────────────────────────────────
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False, header=True)
y_test.to_csv('y_test.csv', index=False, header=True)
X_train_scaled.to_csv('X_train_scaled.csv', index=False)
X_test_scaled.to_csv('X_test_scaled.csv', index=False)
joblib.dump(scaler, 'scaler.pkl')

# ── Save web-app constants (imputation defaults, income transform, age gates) ──
# The web app needs these to correctly pre-process user inputs before prediction.
# Income pre-processing order: signed_log1p FIRST, then scaler.transform().
import json
web_app_constants = {
    'imputation_defaults': {
        'age_median':          age_med,
        'bmi_overall_median':  bmi_overall_med,
        'bmi_child_median':    bmi_child_med,
        'bmi_adult_median':    bmi_adult_med,
        'self_health_median':  sh_med,
        'self_mhealth_median': smh_med,
        'family_size_mode':    fs_mode,
        'k6_adult_median':     float(k6_adult_med),
        'phq2_adult_median':   float(phq2_adult_med),
        'education_mode':      edu_mode,
    },
    'income_transform': {
        'note': 'Apply signed_log1p to family_income and total_person_income BEFORE scaler.transform()',
        'formula': 'np.sign(x) * np.log1p(np.abs(x))',
        'applies_to': ['family_income', 'total_person_income'],
    },
    'age_gates': {
        'bmi_child_range':   [6, 17],   # use child_bmi scale; web app asks BMI for all ages
        'bmi_adult_min':     18,
        'is_student_range':  [17, 23],  # show question only in this range, else = 0
        'mental_health_min': 18,        # k6 + phq2 shown only for 18+, else = 0
        'adult_dx_min':      18,        # 9 adult chronic conditions, else = 0
        'adhd_range':        [5, 17],   # dx_adhd_add shown only in this range, else = 0
        'employment_min':    16,        # employment_status for 16+, else = 0
        'education_min':     5,         # education_years for 5+, else = 0
    },
    'feature_cols': feature_cols,
}
with open('web_app_constants.json', 'w') as f:
    json.dump(web_app_constants, f, indent=2)

print('Saved:')
print('  X_train.csv              X_test.csv')
print('  y_train.csv              y_test.csv')
print('  X_train_scaled.csv       X_test_scaled.csv')
print('  scaler.pkl')
print('  web_app_constants.json   <- imputation defaults, income transform, age gates')

Saved:
  X_train.csv              X_test.csv
  y_train.csv              y_test.csv
  X_train_scaled.csv       X_test_scaled.csv
  scaler.pkl
  web_app_constants.json   <- imputation defaults, income transform, age gates


In [10]:
print('=' * 65)
print('FEATURE ENGINEERING COMPLETE'.center(65))
print('=' * 65)

groups = {
    'Demographics (4)':        ['age', 'sex', 'hispanic', 'bmi'],
    'Socioeconomic (7)':       ['education_years', 'family_size', 'poverty_category',
                                'family_income', 'total_person_income',
                                'employment_status', 'is_student'],
    'Health Status (7)':       ['self_rated_health', 'self_rated_mental_health',
                                'needs_help_iadl', 'needs_help_adl', 'any_limitation',
                                'k6_distress_score', 'phq2_depression_score'],
    'Chronic Conditions (12)': [c for c in feature_cols if c.startswith('dx_')],
    'Insurance (7)':           ['has_private_insurance', 'has_tricare', 'has_medicare',
                                'has_medicaid', 'has_va_coverage', 'is_uninsured',
                                'has_usual_care'],
    'Race one-hot':            [c for c in feature_cols if c.startswith('race_')],
    'Marital one-hot':         [c for c in feature_cols if c.startswith('marital_')],
    'Region one-hot':          [c for c in feature_cols if c.startswith('region_')],
}

total = 0
for group, cols in groups.items():
    available = [c for c in cols if c in feature_cols]
    print(f'  {group:<28} {len(available):>2} features')
    total += len(available)

print('-' * 65)
print(f'  {"TOTAL":<28} {total:>2} features')
print(f'  {"Train rows":<28} {X_train.shape[0]:>6,}')
print(f'  {"Test rows":<28} {X_test.shape[0]:>6,}')
print('=' * 65)
print()
print('Web app input logic:')
print('  - BMI: single question for all ages (uses child_bmi scale for 6-17)')
print('  - is_student: show only when age is 17-23, else default = 0')
print('  - k6 / phq2: show only when age >= 18, else default = 0')
print('  - Adult dx (9 conditions): show only when age >= 18, else default = 0')
print('  - dx_adhd_add: show only when age is 5-17, else default = 0')
print('  - Load scaler.pkl to transform user inputs before Linear/SVR prediction')

                   FEATURE ENGINEERING COMPLETE                  
  Demographics (4)              4 features
  Socioeconomic (7)             7 features
  Health Status (7)             7 features
  Chronic Conditions (12)      12 features
  Insurance (7)                 7 features
  Race one-hot                  8 features
  Marital one-hot               6 features
  Region one-hot                4 features
-----------------------------------------------------------------
  TOTAL                        55 features
  Train rows                   17,944
  Test rows                     4,487

Web app input logic:
  - BMI: single question for all ages (uses child_bmi scale for 6-17)
  - is_student: show only when age is 17-23, else default = 0
  - k6 / phq2: show only when age >= 18, else default = 0
  - Adult dx (9 conditions): show only when age >= 18, else default = 0
  - dx_adhd_add: show only when age is 5-17, else default = 0
  - Load scaler.pkl to transform user inputs before Linear/